# 04 · Explainable AI (XAI)

Why did the models decide what they decided?

**Detection model (EfficientNet-B0)**
- **Grad-CAM** and **Grad-CAM++** heatmaps: which parts of the slice pushed the model towards "tumor".
- **Occlusion sensitivity**: hide one square at a time and measure how much the tumor probability drops.
- **Quantitative check**: what share of the heatmap falls inside the expert tumor outline, and is the hottest point inside the tumor?

**Segmentation model (SegResNet)**
- **Uncertainty maps** from test-time augmentation (predict 8 mirrored versions and measure disagreement):
  where is the model unsure, and do its mistakes happen where it is unsure?

**Needs:** finished runs of notebooks 02 and 03 (same `QUICK_RUN` setting). **Runtime:** T4 GPU.
Everything is saved to `runs/xai/<XAI_RUN_NAME>/`.

In [ ]:
# ======================= SETTINGS - edit these ===========================
PROJECT_DIR = '/content/drive/MyDrive/BrainTumor'
WORK_DIR    = '/content/work'
QUICK_RUN   = False

DET_RUN_NAME = 'effb0_v1'        # detection run to explain (notebook 02)
SEG_RUN_NAME = 'segresnet_v1'    # segmentation run to explain (notebook 03)
XAI_RUN_NAME = 'xai_v1'

N_EXAMPLES          = 8       # example slices per group (true positive / false negative / ...)
N_CAM_SLICES        = 1000    # tumor slices used for the quantitative heatmap check
N_OCCLUSION         = 6       # slices for occlusion maps (slower)
N_UNCERTAINTY_CASES = 20      # test patients for segmentation uncertainty (8 predictions each)
AMP  = True
SEED = 42

In [ ]:
# ---- Setup: mount Google Drive, install missing packages, import the project code ----
import os
import sys

# Only used when testing outside Colab; on Colab these three lines change nothing.
PROJECT_DIR = os.environ.get('BT_PROJECT_DIR', PROJECT_DIR)
WORK_DIR = os.environ.get('BT_WORK_DIR', WORK_DIR)
QUICK_RUN = os.environ.get('BT_QUICK_RUN', '1' if QUICK_RUN else '0') == '1'

if PROJECT_DIR.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

CODE_DIR = os.path.join(PROJECT_DIR, 'code')
if not os.path.isfile(os.path.join(CODE_DIR, 'brats', '__init__.py')):
    raise FileNotFoundError(f'The "brats" folder was not found in {CODE_DIR}.\n'
                            'Upload everything from the "Code" folder to that Drive folder first (see README).')
if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

from brats.notebook import install_missing
install_missing({'nibabel': 'nibabel', 'monai': 'monai', 'timm': 'timm', 'sklearn': 'scikit-learn'})

import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import roc_auc_score, roc_curve
from tqdm.auto import tqdm

from brats import config, dataset, detection, env, fileio, metrics, plots, segmentation, xai
from brats.case import region_masks

if QUICK_RUN:
    N_EXAMPLES, N_CAM_SLICES, N_OCCLUSION, N_UNCERTAINTY_CASES = 4, 40, 2, 2
paths = env.ProjectPaths(PROJECT_DIR, WORK_DIR, quick=QUICK_RUN).make()
env.print_header('QUICK RUN - smoke test only' if QUICK_RUN else f'XAI RUN "{XAI_RUN_NAME}"')
device = env.get_device()
env.require_gpu(device, QUICK_RUN)
env.set_seed(SEED)
det_dir = paths.run_path('detection', DET_RUN_NAME)
seg_dir = paths.run_path('segmentation', SEG_RUN_NAME)
for run_dir, notebook in ((det_dir, '02'), (seg_dir, '03')):
    if not (run_dir / 'checkpoints' / 'best.pt').is_file() or not (run_dir / 'summary.json').is_file():
        raise FileNotFoundError(f'{run_dir} has no finished run. Run notebook {notebook} to the end first '
                                '(same QUICK_RUN).')
xai_dir = paths.run_dir('xai', XAI_RUN_NAME)
FIG = xai_dir / 'figures'
fileio.save_json(dict(det_run=str(det_dir), seg_run=str(seg_dir), n_examples=N_EXAMPLES, n_cam_slices=N_CAM_SLICES,
                      n_occlusion=N_OCCLUSION, n_uncertainty_cases=N_UNCERTAINTY_CASES, seed=SEED,
                      quick_run=QUICK_RUN), xai_dir / 'config.json')
env.save_environment(xai_dir / 'logs')
env.snapshot_code(xai_dir, paths.code, '04_xai.ipynb')
print('device', device, '| results ->', xai_dir)

In [ ]:
cases_dir = dataset.stage_to_local_disk(paths)
cases, slices, split = dataset.load_metadata(paths)
test_ids = dataset.split_case_ids(split)['test']
dataset.verify_staged(cases_dir, test_ids)

det_model, det_card = detection.load_detector(det_dir, device)
threshold = det_card['threshold']
test_predictions = pd.read_csv(det_dir / 'predictions' / 'test_slices.csv')
print(f'Detection model {det_card["model_name"]}, threshold {threshold:.3f}, {len(test_predictions)} test slices')

## 1 · Grad-CAM and Grad-CAM++ on example slices

In [ ]:
def load_slices(rows: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Model inputs (K, 4, 240, 240) and expert tumor masks (K, 240, 240) for rows of a predictions table, in row order."""
    inputs, masks = [None] * len(rows), [None] * len(rows)
    position = {index: k for k, index in enumerate(rows.index)}
    for case_id, group in rows.groupby('case_id'):
        case = dataset.load_case(cases_dir, case_id)
        case_inputs = detection.detection_inputs(case, group['z'].values)
        case_masks = detection.detection_masks(case, group['z'].values)
        for j, index in enumerate(group.index):
            inputs[position[index]], masks[position[index]] = case_inputs[j], case_masks[j]
    return np.stack(inputs), np.stack(masks)


def heatmaps(inputs: np.ndarray, plus_plus: bool = False, batch: int = 32) -> tuple[np.ndarray, np.ndarray]:
    """Grad-CAM (or Grad-CAM++) heatmaps and tumor probabilities, in batches."""
    cams, probs = [], []
    for start in range(0, len(inputs), batch):
        cam, prob = xai.gradcam(det_model, torch.from_numpy(inputs[start:start + batch]).float().to(device), plus_plus)
        cams.append(cam)
        probs.append(prob)
    return np.concatenate(cams), np.concatenate(probs)


groups = {'true_positive': test_predictions[(test_predictions['label'] == 1) & (test_predictions['pred'] == 1)],
          'false_negative': test_predictions[(test_predictions['label'] == 1) & (test_predictions['pred'] == 0)],
          'false_positive': test_predictions[(test_predictions['label'] == 0) & (test_predictions['pred'] == 1)],
          'true_negative': test_predictions[(test_predictions['label'] == 0) & (test_predictions['pred'] == 0)]}
for i, (group_name, group_rows) in enumerate(groups.items()):
    if len(group_rows) == 0:
        print(f'{group_name}: no examples')
        continue
    chosen = group_rows.sample(n=min(N_EXAMPLES, len(group_rows)), random_state=SEED).reset_index(drop=True)
    inputs, masks = load_slices(chosen)
    cam, prob = heatmaps(inputs)
    cam_pp, _ = heatmaps(inputs, plus_plus=True)
    images, heats, outlines, titles = [], [], [], []
    for k in range(len(chosen)):
        flair = plots.view2d(inputs[k, 0])
        outline = [(plots.view2d(masks[k]), plots.PALETTE[2])]
        images += [flair, flair, flair]
        heats += [None, plots.view2d(cam[k]), plots.view2d(cam_pp[k])]
        outlines += [outline, outline, outline]
        titles += [f'FLAIR p={prob[k]:.2f}', 'Grad-CAM', 'Grad-CAM++']
    fig = plots.image_grid(images, titles, contours=outlines, heats=heats, ncols=6, cell=2.0,
                           suptitle=f'{group_name.replace("_", " ").capitalize()}s - green outline = expert tumor')
    plots.save_fig(fig, FIG, f'{i + 1:02d}_gradcam_{group_name}', data=chosen)
    fileio.save_npz(xai_dir / f'gradcam_{group_name}.npz', cam=cam.astype(np.float16), campp=cam_pp.astype(np.float16),
                    prob=prob, case_id=chosen['case_id'].values.astype(str), z=chosen['z'].values)

## 2 · Does the heatmap point at the tumor? (quantitative)
- **Energy inside tumor**: share of the heatmap that lies inside the expert outline (compared with the share of the brain the tumor covers,
  which is what a random heatmap would get).
- **Pointing game**: is the hottest pixel inside the tumor?

In [ ]:
tumor_rows = test_predictions[test_predictions['label'] == 1]
tumor_rows = tumor_rows.sample(n=min(N_CAM_SLICES, len(tumor_rows)), random_state=SEED).reset_index(drop=True)
rows = []
for start in tqdm(range(0, len(tumor_rows), 64), desc='heatmaps'):
    chunk = tumor_rows.iloc[start:start + 64].reset_index(drop=True)
    inputs, masks = load_slices(chunk)
    cam, prob = heatmaps(inputs)
    cam_pp, _ = heatmaps(inputs, plus_plus=True)
    brain = np.any(inputs != 0, axis=1)
    for k in range(len(chunk)):
        peak = np.unravel_index(np.argmax(cam[k]), cam[k].shape)
        peak_pp = np.unravel_index(np.argmax(cam_pp[k]), cam_pp[k].shape)
        rows.append({'case_id': chunk.loc[k, 'case_id'], 'z_orig': chunk.loc[k, 'z_orig'], 'prob': prob[k],
                     'detected': prob[k] >= threshold, 'tumor_px': int(masks[k].sum()),
                     'tumor_share_of_brain': masks[k].sum() / max(brain[k].sum(), 1),
                     'gradcam_energy_inside': xai.heatmap_share_inside(cam[k], masks[k]),
                     'gradcampp_energy_inside': xai.heatmap_share_inside(cam_pp[k], masks[k]),
                     'gradcam_peak_inside': bool(masks[k][peak]), 'gradcampp_peak_inside': bool(masks[k][peak_pp])})
cam_table = pd.DataFrame(rows)
fileio.save_csv(cam_table, xai_dir / 'heatmap_quantitative.csv')
heatmap_summary = {'n_slices': len(cam_table),
                   'gradcam_energy_inside_mean': float(cam_table['gradcam_energy_inside'].mean()),
                   'gradcampp_energy_inside_mean': float(cam_table['gradcampp_energy_inside'].mean()),
                   'random_baseline_mean': float(cam_table['tumor_share_of_brain'].mean()),
                   'gradcam_pointing_game': float(cam_table['gradcam_peak_inside'].mean()),
                   'gradcampp_pointing_game': float(cam_table['gradcampp_peak_inside'].mean())}
display(pd.Series(heatmap_summary).round(4))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
edges = np.linspace(0, 1, 26)
axes[0].hist(cam_table['tumor_share_of_brain'], bins=edges, color=plots.MUTED, alpha=0.6,
             label='random heatmap (tumor share of brain)')
axes[0].hist(cam_table['gradcam_energy_inside'].dropna(), bins=edges, color=plots.PALETTE[0], alpha=0.75,
             label='Grad-CAM')
axes[0].hist(cam_table['gradcampp_energy_inside'].dropna(), bins=edges, color=plots.PALETTE[1], alpha=0.6,
             label='Grad-CAM++')
axes[0].set_xlabel('share of heatmap inside the expert tumor')
axes[0].set_ylabel('slices')
axes[0].legend(fontsize=8)
axes[0].set_title('Heatmap energy inside the tumor')
axes[1].scatter(cam_table['tumor_px'], cam_table['gradcam_energy_inside'], s=8, alpha=0.5, color=plots.PALETTE[0],
                linewidths=0)
plots.log_axis(axes[1], 'x')
axes[1].set_xlabel('tumor pixels on the slice (log)')
axes[1].set_ylabel('Grad-CAM energy inside')
axes[1].set_title('Larger tumors are easier to point at')
plots.bar_panel(axes[2], ['Grad-CAM', 'Grad-CAM++', 'random'],
                [100 * heatmap_summary['gradcam_pointing_game'], 100 * heatmap_summary['gradcampp_pointing_game'],
                 100 * heatmap_summary['random_baseline_mean']],
                colors=[plots.PALETTE[0], plots.PALETTE[1], plots.MUTED],
                title='Pointing game: hottest pixel inside tumor', ylabel='% of slices', fmt='{:.0f}%')
fig.tight_layout()
plots.save_fig(fig, FIG, '05_heatmap_quantitative', data=cam_table)

## 3 · Occlusion sensitivity vs Grad-CAM

In [ ]:
true_positives = groups['true_positive']
occlusion_rows = []
if len(true_positives):
    chosen = (true_positives.sort_values('prob', ascending=False).head(max(N_OCCLUSION * 5, N_OCCLUSION))
              .sample(n=min(N_OCCLUSION, len(true_positives)), random_state=SEED).reset_index(drop=True))
    inputs, masks = load_slices(chosen)
    cam, _ = heatmaps(inputs)
    images, heats, outlines, titles = [], [], [], []
    for k in range(len(chosen)):
        drop, base_prob = xai.occlusion_map(det_model, inputs[k], device, patch=24, stride=12)
        drop_scaled = np.clip(drop / max(drop.max(), 1e-6), 0, 1)
        flair = plots.view2d(inputs[k, 0])
        outline = [(plots.view2d(masks[k]), plots.PALETTE[2])]
        images += [flair, flair, flair]
        heats += [None, plots.view2d(cam[k]), plots.view2d(drop_scaled)]
        outlines += [outline, outline, outline]
        titles += [f'FLAIR p={base_prob:.2f}', 'Grad-CAM', 'Occlusion']
        occlusion_rows.append({'case_id': chosen.loc[k, 'case_id'], 'z_orig': chosen.loc[k, 'z_orig'],
                               'prob': base_prob,
                               'occlusion_energy_inside': xai.heatmap_share_inside(np.clip(drop, 0, None), masks[k]),
                               'gradcam_energy_inside': xai.heatmap_share_inside(cam[k], masks[k]),
                               'max_prob_drop': float(drop.max())})
        np.save(xai_dir / f'occlusion_{chosen.loc[k, "case_id"]}_z{chosen.loc[k, "z_orig"]}.npy', drop)
    fig = plots.image_grid(images, titles, contours=outlines, heats=heats, ncols=6, cell=2.0,
                           suptitle='Occlusion: hiding the tumor lowers the tumor probability (green = expert tumor)')
    plots.save_fig(fig, FIG, '06_occlusion_vs_gradcam', data=pd.DataFrame(occlusion_rows))

## 4 · Segmentation uncertainty (test-time augmentation)
Each test patient is predicted 8 times (all mirror combinations). Where the 8 predictions disagree, the model is uncertain.
A trustworthy model is uncertain where it makes mistakes.

In [ ]:
seg_model, seg_card = segmentation.load_segmenter(seg_dir, device)
seg_config = fileio.load_json(seg_dir / 'config.json')
roi, seg_threshold = tuple(seg_card['roi']), seg_card['threshold']
sw_batch, sw_overlap = seg_config['sw_batch'], seg_card['sw_overlap']
rng = np.random.default_rng(SEED)
uncertainty_ids = sorted(rng.choice(test_ids, size=min(N_UNCERTAINTY_CASES, len(test_ids)), replace=False))
uncertainty_rows, voxel_uncertainty, voxel_error, examples = [], [], [], []
for case_id in tqdm(uncertainty_ids, desc='uncertainty'):
    case = dataset.load_case(cases_dir, case_id)
    image = torch.from_numpy(case.normalized())
    start = time.time()
    single = segmentation.predict_volume(seg_model, image, device, roi, sw_batch, sw_overlap, AMP)
    mean, std = segmentation.predict_volume_tta(seg_model, image, device, roi, sw_batch, sw_overlap, AMP)
    seconds = time.time() - start
    truth = case.labels
    labels_single = segmentation.probs_to_labels(single, seg_threshold)
    labels_tta = segmentation.probs_to_labels(mean, seg_threshold)
    masks_single, masks_tta = region_masks(labels_single), region_masks(labels_tta)
    masks_true = region_masks(truth)
    entropy = xai.binary_entropy(mean[0])                    # uncertainty of the whole-tumor channel
    error = masks_tta['WT'] != masks_true['WT']
    tumor_area = masks_tta['WT'] | masks_true['WT']
    row = {'case_id': case_id, 'seconds_9_predictions': seconds,
           'mean_entropy_in_tumor_area': float(entropy[tumor_area].mean()) if tumor_area.any() else 0.0,
           'mean_std_in_tumor_area': float(std[0][tumor_area].mean()) if tumor_area.any() else 0.0,
           'error_voxels': int(error.sum()), 'uncertain_voxels_entropy_gt_0.5': int((entropy > 0.5).sum())}
    for region in config.REGIONS:
        row[f'dice_single_{region}'] = metrics.dice_score(masks_single[region], masks_true[region])
        row[f'dice_tta_{region}'] = metrics.dice_score(masks_tta[region], masks_true[region])
    uncertainty_rows.append(row)
    brain_voxels = np.flatnonzero(case.brain_mask())
    sampled = rng.choice(brain_voxels, size=min(200_000, brain_voxels.size), replace=False)
    voxel_uncertainty.append(entropy.ravel()[sampled])
    voxel_error.append(error.ravel()[sampled])
    fileio.save_npz(xai_dir / f'uncertainty_{case_id}.npz', mean=mean.astype(np.float16), std=std.astype(np.float16),
                    entropy_wt=entropy.astype(np.float16), bbox=case.bbox)
    if len(examples) < 4:
        k = int(np.argmax(tumor_area.sum((0, 1)))) if tumor_area.any() else truth.shape[2] // 2
        examples.append((case_id, image[0, :, :, k].numpy(), truth[:, :, k], labels_tta[:, :, k], error[:, :, k],
                         entropy[:, :, k]))
uncertainty_table = pd.DataFrame(uncertainty_rows)
fileio.save_csv(uncertainty_table, xai_dir / 'uncertainty_per_patient.csv')
voxel_uncertainty, voxel_error = np.concatenate(voxel_uncertainty), np.concatenate(voxel_error)
try:
    uncertainty_auc = float(roc_auc_score(voxel_error, voxel_uncertainty))
except ValueError:          # no errors at all (or only errors) in the sample
    uncertainty_auc = float('nan')
print(f'Uncertainty as an error detector (voxel AUC): {uncertainty_auc:.3f}  (0.5 = useless, 1.0 = perfect)')
display(uncertainty_table.round(3))

In [ ]:
fig, axes = plt.subplots(len(examples), 5, figsize=(13, 2.7 * max(len(examples), 1)), squeeze=False)
for row, (case_id, flair, truth, pred, error, entropy) in enumerate(examples):
    first = row == 0
    plots.show_gray(axes[row, 0], plots.view2d(flair), 'FLAIR' if first else None)
    plots.show_labels(axes[row, 1], plots.view2d(flair), plots.view2d(truth), 'Expert' if first else None)
    plots.show_labels(axes[row, 2], plots.view2d(flair), plots.view2d(pred), 'Model (8x TTA)' if first else None)
    plots.show_heat(axes[row, 3], plots.view2d(flair), plots.view2d(error.astype(float)), 'Errors' if first else None)
    plots.show_heat(axes[row, 4], plots.view2d(flair), plots.view2d(entropy), 'Uncertainty (entropy)' if first else None)
    axes[row, 0].text(-0.08, 0.5, case_id[-5:], transform=axes[row, 0].transAxes, rotation=90, va='center', ha='right',
                      fontsize=8)
plots.label_legend(fig)
fig.suptitle('Where the segmentation model is unsure, and where it is wrong', y=1.0)
fig.tight_layout(rect=(0, 0.04, 1, 1))
plots.save_fig(fig, FIG, '07_uncertainty_examples')

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
edges = np.linspace(0, 1, 41)
axes[0].hist(voxel_uncertainty[~voxel_error], bins=edges, color=plots.PALETTE[0], alpha=0.75, label='correct voxels',
             density=True)
axes[0].hist(voxel_uncertainty[voxel_error], bins=edges, color=plots.PALETTE[1], alpha=0.7, label='wrong voxels',
             density=True)
plots.log_axis(axes[0], 'y')
axes[0].set_xlabel('uncertainty (entropy, bits)')
axes[0].set_ylabel('density (log)')
axes[0].set_title('Mistakes happen where the model is unsure')
axes[0].legend()
if np.isfinite(uncertainty_auc):
    fpr, tpr, _ = roc_curve(voxel_error, voxel_uncertainty)
    axes[1].plot(fpr, tpr, color=plots.PALETTE[0], label=f'AUC {uncertainty_auc:.3f}')
axes[1].plot([0, 1], [0, 1], color=plots.AXIS, linestyle='--', linewidth=1)
axes[1].set_xlabel('false alarm rate')
axes[1].set_ylabel('share of errors flagged')
axes[1].set_title('Uncertainty as an error detector')
axes[1].legend(loc='lower right')
axes[2].scatter(uncertainty_table['mean_entropy_in_tumor_area'], uncertainty_table['dice_tta_WT'], s=18,
                color=plots.PALETTE[0], alpha=0.8, linewidths=0)
axes[2].set_xlabel('mean uncertainty in the tumor area')
axes[2].set_ylabel('Dice WT')
axes[2].set_title('Uncertain patients get lower Dice')
fig.tight_layout()
plots.save_fig(fig, FIG, '08_uncertainty_vs_errors', data=uncertainty_table)

fig, ax = plt.subplots(figsize=(7, 4))
plots.box_panel(ax, [uncertainty_table[f'dice_single_{r}'] for r in config.REGIONS]
                + [uncertainty_table[f'dice_tta_{r}'] for r in config.REGIONS],
                [f'{r}\nsingle' for r in config.REGIONS] + [f'{r}\n8x TTA' for r in config.REGIONS],
                colors=[plots.REGION_COLORS[r] for r in config.REGIONS] * 2, ylabel='Dice',
                title='Does averaging 8 mirrored predictions help?')
fig.tight_layout()
plots.save_fig(fig, FIG, '09_tta_vs_single_prediction', data=uncertainty_table)

summary = {'detection_run': det_dir.name, 'segmentation_run': seg_dir.name, 'heatmaps': heatmap_summary,
           'occlusion': pd.DataFrame(occlusion_rows).mean(numeric_only=True).to_dict() if occlusion_rows else {},
           'uncertainty_error_auc': uncertainty_auc,
           'tta_dice_gain': {r: float((uncertainty_table[f'dice_tta_{r}'] - uncertainty_table[f'dice_single_{r}']).mean())
                             for r in config.REGIONS}}
fileio.save_json(summary, xai_dir / 'summary.json')
print('Saved', xai_dir / 'summary.json')

## Done
Saved in `runs/xai/<XAI_RUN_NAME>/`: heatmap figures, quantitative CSVs, occlusion maps, uncertainty maps per patient,
`summary.json`, `logs/`, `code_snapshot/`.

In [ ]:
env.flush_drive()